# Day 88: MoE Routing from Scratch

**Layer:** Advanced


## Concept Overview

Mixture-of-Experts replaces each FFN layer with E experts and a router that sends each token to the top-K experts. The load balancing loss penalizes routers that consistently route all tokens to a few popular experts.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Top-K Gating with Load Balancing

Implement top-2 gating, compute expert utilization, and add auxiliary load balancing loss.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MoELayer(nn.Module):
    def __init__(self, d_model=512, n_experts=8, top_k=2, d_ff=2048):
        super().__init__()
        self.n_experts = n_experts
        self.top_k = top_k
        self.router = nn.Linear(d_model, n_experts, bias=False)
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
            for _ in range(n_experts)])
    
    def forward(self, x):
        B, T, D = x.shape
        logits = self.router(x.view(-1, D))  # (B*T, E)
        weights = F.softmax(logits, dim=-1)
        top_weights, top_experts = torch.topk(weights, self.top_k, dim=-1)
        top_weights = top_weights / top_weights.sum(dim=-1, keepdim=True)
        
        # Load balancing loss
        expert_usage = (top_experts == torch.arange(self.n_experts).unsqueeze(0)).float().mean(0)
        lb_loss = (expert_usage * weights.mean(0)).sum() * self.n_experts
        
        # Dispatch (simplified: no expert parallelism)
        out = torch.zeros_like(x.view(-1, D))
        for e in range(self.n_experts):
            mask = (top_experts == e)
            if mask.any():
                tokens = (x.view(-1,D).unsqueeze(1).expand(-1,self.top_k,-1)[mask])
                out[mask.any(-1)] += self.experts[e](tokens) * top_weights[mask].unsqueeze(-1)
        return out.view(B,T,D), lb_loss

moe = MoELayer()
x = torch.randn(2, 16, 512)
y, lb_loss = moe(x)
print(f"Input: {x.shape} -> Output: {y.shape}")
print(f"Load balancing loss: {lb_loss.item():.4f} (target: 1.0 = uniform)")


## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- Mixture-of-Experts replaces each FFN layer with E experts and a router that sends each token to the top-K experts.
- Mixture-of-Experts replaces each FFN layer with E experts and a router that send....
- Day 88 implementation complete.
